# 课后练习解答（02.07_evaluation_and_inference）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** logits 形状为 (256, 200)，torch.topk(logits, 5, dim=1) 返回的两个张量形状是？
A. values(256,5) 与 indices(256,5)
B. values(5,256) 与 indices(5,256)
C. 只有一个张量 (256,5)
D. (256,200) 与 (256,200)

**解答：** A

**解析：** topk 沿 dim=1 取每行前 5 个，返回 values 与 indices 两个同形张量。


### 问题2（单选题）

**题目：** Top-5 准确率的计算方式是？
A. 真实类别是否出现在模型概率最高的前 5 个类别中
B. 前 5 个样本预测正确
C. 5 次推理取平均
D. 类别数不超过 5

**解答：** A

**解析：** Top-5 是每样本判断真实标签是否在 top-5 候选内，再统计正确比例。


### 问题3（多选题）

**题目：** 为保证评估结果可复现，需要固定？
A. 与训练一致的 Normalize/尺寸
B. 同一份 checkpoint 与 dtype
C. 固定随机种子（若推理含随机操作）
D. 训练集增强

**解答：** ABC

**解析：** 训练集增强不参与评估，不会影响验证指标可复现性。


### 问题4（多选题）

**题目：** 混淆矩阵可以用于？
A. 发现高频混淆类别对
B. 计算每类召回率
C. 定位具体错误样本
D. 替代 Top-1 指标

**解答：** ABC

**解析：** 混淆矩阵是分析工具，不能替代总体准确率指标。


### 问题5（判断题）

**题目：** 同一个模型在相同数据上，Top-5 准确率一定不低于 Top-1。

**解答：** 对

**解析：** Top-1 正确必然满足 Top-5 条件，因此 Top-5 是 Top-1 的超集。


### 问题6（判断题）

**题目：** 推理时应沿用训练时的 RandAugment 以保证数据分布一致。

**解答：** 错

**解析：** 推理应使用确定性预处理，随机增强会破坏评估可复现性。


### 问题7（填空题）

**题目：** torch.max(logits, dim=1) 返回两个张量：最大概率值与对应的 ____。

**解答：** 类别索引（indices）


### 问题8（填空题）

**题目：** 从 checkpoint 恢复推理前，必须先按相同结构创建模型，再调用 ____ 加载权重。

**解答：** load_state_dict()


### 问题9（简答题）

**题目：** 为什么批量评估通常比逐张图片评估更高效？请结合算子调度与访存说明。

**解答：** 批量推理让矩阵乘与卷积按大 tensor 调度，减少 kernel 启动次数和算子间等待；同时访存局部性更好，设备利用率更高。


### 问题10（简答题）

**题目：** 如何使用混淆矩阵分析某一类别被系统性地误判为另一类别的问题？

**解答：** 统计混淆矩阵第 i 行中非对角元素，找出占比最高的列 j；抽样查看真实 i 类被预测为 j 类的样本，判断是视觉相似、标注错误还是数据不足，再决定是否增加样本或合并细粒度类别。


### 问题11（代码设计题）

**题目：** 编写 evaluate(model, loader, device)，返回 acc1、acc5 与宏平均召回率，要求 eval/no_grad。

**解答：** ```python
def evaluate(model, loader, device):
    model.eval()
    correct1 = correct5 = total = 0
    per_class_correct = {}
    per_class_total = {}
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            _, pred = logits.topk(5, 1, True, True)
            correct1 += (pred[:, :1] == labels.view(-1, 1)).sum().item()
            correct5 += (pred == labels.view(-1, 1)).sum().item()
            total += labels.size(0)
            for l, p in zip(labels.tolist(), pred[:, 0].tolist()):
                per_class_total[l] = per_class_total.get(l, 0) + 1
                per_class_correct[l] = per_class_correct.get(l, 0) + (1 if l == p else 0)
    acc1 = correct1 / total
    acc5 = correct5 / total
    recall = sum(per_class_correct.get(c, 0) / n for c, n in per_class_total.items()) / len(per_class_total)
    return acc1, acc5, recall
```


### 问题12（单选题）

**题目：** 评估时 logits 中出现 NaN，最合理的处理是？
A. 先检查 loss/预处理/数值稳定性，修复后再评估
B. 直接取 NaN 对应索引
C. 忽略 NaN
D. 将 NaN 替换为 0

**解答：** A

**解析：** NaN 是数值链路故障信号，任何后续指标都不可信。


### 问题13（多选题）

**题目：** 类别不均衡时，更适合使用的指标包括？
A. 宏平均召回率
B. 各类 F1
C. 只看 Top-1
D. 混淆矩阵

**解答：** ABD

**解析：** 总体 Top-1 会被大类主导，掩盖小类性能。


### 问题14（判断题）

**题目：** 计算验证准确率时应将模型保持为 train 模式，避免 BN 统计量偏差。

**解答：** 错

**解析：** 验证必须使用 eval 模式，BN 使用 running 统计量，Dropout 关闭。


### 问题15（简答题）

**题目：** 在 Tiny ImageNet 中，为什么 Top-5 对相似类别任务更友好？请结合类别粒度说明。

**解答：** Tiny ImageNet 包含大量细粒度相似类别，模型可能把真实类别放在第 2~5 位；Top-5 能反映模型语义空间已经接近正确答案，避免因“差一点”被判全错。
